# Quest Hero: Solutions Notebook

This notebook contains complete solutions for both TODOs.

In [29]:
# ── Setup (run this first) ──────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_hero")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

Setup complete ✓


In [30]:
from quest_hero import *
from quest_hero.game_world import GameWorld, CellType, NPC, NPC_CATALOG
from quest_hero.hero import Hero
from quest_hero.agent import TOOLS_DESCRIPTION, parse_tool_call
from quest_hero.oracle import ORACLE_TEMPLATE, llm_oracle
from quest_hero.display import render_grid
print("Quest Hero loaded!")

Quest Hero loaded!


In [31]:
# Show the full map
hero, world, tools = create_game()
print(render_grid(world, hero, show_all=True))

     0    1    2    3    4    5    6    7 
   ---------------------------------
0  | 🏰  | ·  | 🌲  | ·  | ·  | 🌲  | ·  | 💎  |
   ---------------------------------
1  | ·  | ·  | ·  | ⛰️ | ·  | ·  | ·  | ·  |
   ---------------------------------
2  | 🌲  | 🐉  | ·  | ⛰️ | ·  | 🏚️ | ·  | 🌲  |
   ---------------------------------
3  | ·  | ·  | ·  | ·  | 🧙  | ·  | ·  | ·  |
   ---------------------------------
4  | ·  | 🏪  | ·  | ⛰️ | ·  | ·  | 🐉  | ·  |
   ---------------------------------
5  | 💎  | ·  | ·  | 🧙  | ·  | 🌲  | ·  | ·  |
   ---------------------------------
6  | ·  | ·  | 🏪  | ·  | ·  | ·  | 💎  | ·  |
   ---------------------------------
7  | 🦸 | 🧙  | ·  | ·  | 🏚️ | ·  | ·  | 🌲  |
   ---------------------------------


---
## Solution: TODO 1 — Rule-Based Think Function

Strategy: systematic exploration with priority-based actions.
- Pick up items on current cell
- Talk to NPCs/innkeepers/shopkeepers
- Buy/craft when we have the materials
- Fight dragons when we have the right weapon
- Move toward unvisited cells, preferring cells with interesting content

In [32]:
from collections import deque

def _bfs_next_step(start, target, world):
    """BFS pathfinding that navigates around mountains."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def think_rule_based(hero: Hero, world: GameWorld, history: list[dict]) -> str:
    row, col = hero.position
    cell = world.get_cell(row, col)

    # --- Priority 1: Pick up items on current cell ---
    if cell.items:
        return 'TOOL: pick_up()'

    # --- Priority 2: Interact with current cell ---
    if cell.cell_type == CellType.NPC and cell.npc:
        npc_name = cell.npc.name
        talked = sum(1 for e in hero.journal if npc_name in e)
        if talked == 0:
            return 'TOOL: talk(question="What can you tell me? What quests or dangers should I know about?")'
        if cell.npc_id == "traveling_merchant" and not hero.has_item("Letter") and not world.errand_letter_picked_up:
            return 'TOOL: talk(question="Do you have any letter or delivery job for me?")'
        if cell.npc_id == "forest_witch" and hero.has_item("Ghostcaps"):
            return 'TOOL: talk(question="I have Ghostcaps. Can you brew medicine?")'
        if cell.npc_id == "old_hermit" and hero.has_item("Letter"):
            return 'TOOL: talk(question="I have a letter for you from the merchant.")'

    if cell.cell_type == CellType.INN:
        if hero.position == (2, 5):
            if hero.has_item("Herbal Tea"):
                return 'TOOL: talk(question="I brought herbal tea from your brother!")'
            if hero.has_item("Medicine"):
                return 'TOOL: talk(question="I have medicine for your wife!")'
        if hero.position == (7, 4) and not world.errand_tea_picked_up:
            return 'TOOL: talk(question="Hello, do you need any help?")'
        if hero.hearts < hero.MAX_HEARTS and hero.gold >= 2:
            return 'TOOL: rest()'

    if cell.cell_type == CellType.SHOP:
        if hero.has_item("Ember Ore") and hero.gold >= 1 and hero.position == (6, 2):
            return 'TOOL: buy(item="Sunblade")'
        if hero.has_item("Moonstone") and hero.gold >= 1 and hero.position == (4, 1):
            return 'TOOL: buy(item="Moonstone Lantern")'
        if hero.has_item("Dragon Scales") and hero.position == (4, 1):
            return 'TOOL: buy(item="sell Dragon Scales")'

    if cell.cell_type == CellType.DRAGON:
        if cell.dragon_name == "Frost Wyrm" and hero.has_item("Sunblade") and world.frost_wyrm_alive:
            return 'TOOL: fight()'
        if cell.dragon_name == "Shadow Beast" and hero.has_item("Moonstone Lantern") and world.shadow_beast_alive:
            return 'TOOL: fight()'

    if hero.hearts <= 1:
        if hero.has_item("Health Potion"):
            return 'TOOL: use(item="Health Potion")'
        if hero.has_item("Bread"):
            return 'TOOL: use(item="Bread")'

    # --- Priority 3: Navigate to targets using BFS ---
    targets = []

    if hero.has_item("Letter"):
        targets.append((7, 1))
    if hero.has_item("Herbal Tea"):
        targets.append((2, 5))
    if hero.has_item("Medicine"):
        targets.append((2, 5))
    if hero.has_item("Ember Ore"):
        targets.append((6, 2))
    if hero.has_item("Sunblade") and world.frost_wyrm_alive:
        targets.append((2, 1))
    if hero.has_item("Moonstone"):
        targets.append((4, 1))
    if hero.has_item("Moonstone Lantern") and world.shadow_beast_alive:
        targets.append((4, 6))
    if hero.has_item("Dragon Scales"):
        targets.append((4, 1))

    for tc in [(5, 0), (6, 6), (0, 7)]:
        if tc not in hero.visited:
            targets.append(tc)

    if (7, 1) not in hero.visited:
        targets.append((7, 1))
    if (7, 4) not in hero.visited:
        targets.append((7, 4))
    if (5, 3) not in hero.visited:
        targets.append((5, 3))
    if (3, 4) not in hero.visited:
        targets.append((3, 4))

    if not hero.has_item("Ember Ore") and not hero.has_item("Sunblade"):
        if (2, 0) not in hero.visited:
            targets.append((2, 0))

    seen = set()
    unique = []
    for t in targets:
        if t not in seen and t != (row, col):
            seen.add(t)
            unique.append(t)

    for target in unique:
        d = _bfs_next_step((row, col), target, world)
        if d:
            return f'TOOL: move(direction="{d}")'

    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if (nr, nc) not in hero.visited and world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'

    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'

    return 'TOOL: look()'

In [33]:
result = play_rule_based(think_rule_based, max_turns=100, delay=0.05)
print(f"\nResult: {'Victory!' if result['won'] else 'Defeat...'} | Gold: {result['gold']} | Hearts: {result['hearts']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")


Result: Victory! | Gold: 10 | Hearts: 3 | Turns: 83
Game log saved to: game_log_victory_20260312_105045.json


---
## Solution: TODO 2 — LLM Think Function

In [34]:
import os
from google import genai

# os.environ["GEMINI_API_KEY"] = "your-key-here"
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

Gemini client ready!


In [35]:
import re
from collections import deque

# ── Helper: BFS pathfinding ──────────────────────────────────────────────────

def _bfs_next_step_llm(start, target, world):
    """BFS pathfinding that navigates around mountains."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) == target:
            return path[0] if path else None
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _bfs_nearest_unvisited(start, hero, world):
    """BFS to find direction toward the nearest unvisited passable cell."""
    queue = deque([(start, [])])
    visited = {start}
    while queue:
        (r, c), path = queue.popleft()
        if (r, c) not in hero.visited and path:
            return path[0]
        for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
            nr, nc = r + dr, c + dc
            if (nr, nc) not in visited and world.is_passable(nr, nc):
                visited.add((nr, nc))
                queue.append(((nr, nc), path + [d]))
    return None


def _get_quest_targets(hero, world):
    """Return a list of (row, col) targets based on current active quests.
    
    These are derived from what the agent has discovered through gameplay —
    items in inventory imply quests that were triggered by NPC conversations.
    """
    targets = []
    if hero.has_item("Letter"):
        targets.append((7, 1))  # Old Hermit
    if hero.has_item("Herbal Tea"):
        targets.append((2, 5))  # Northern Inn
    if hero.has_item("Medicine"):
        targets.append((2, 5))  # Northern Inn
    if hero.has_item("Ember Ore") and not hero.has_item("Sunblade"):
        targets.append((6, 2))  # Blacksmith
    if hero.has_item("Sunblade") and world.frost_wyrm_alive:
        targets.append((2, 1))  # Frost Wyrm
    if hero.has_item("Moonstone") and not hero.has_item("Moonstone Lantern"):
        targets.append((4, 1))  # Enchanter
    if hero.has_item("Moonstone Lantern") and world.shadow_beast_alive:
        targets.append((4, 6))  # Shadow Beast
    if hero.has_item("Dragon Scales"):
        targets.append((4, 1))  # Enchanter
    return targets


def _bfs_navigate(hero, world):
    """BFS navigation toward quest targets, then nearest unvisited cell.
    
    This is the key architectural insight: navigation is a deterministic problem
    that LLMs are bad at. BFS always finds the shortest path. We let the LLM
    handle decisions (what to say, which quest to pursue) but never ask it
    to figure out cardinal directions from grid coordinates.
    """
    row, col = hero.position

    # Priority 1: BFS toward active quest targets
    for target in _get_quest_targets(hero, world):
        d = _bfs_next_step_llm((row, col), target, world)
        if d:
            return f'TOOL: move(direction="{d}")'

    # Priority 2: BFS to nearest unvisited cell (exploration)
    d = _bfs_nearest_unvisited((row, col), hero, world)
    if d:
        return f'TOOL: move(direction="{d}")'

    # Priority 3: any passable direction
    for d, (dr, dc) in [("north",(-1,0)),("south",(1,0)),("east",(0,1)),("west",(0,-1))]:
        nr, nc = row + dr, col + dc
        if world.is_passable(nr, nc):
            return f'TOOL: move(direction="{d}")'
    return 'TOOL: move(direction="north")'


# ── Agentic AI Pattern 1: Autopilot for Obvious Interactions ────────────────
# Before consulting the LLM, check if the current cell has an obvious action.
# This prevents the LLM from "walking past" NPCs, treasures, and shops.
# It only fires when the agent is already standing on the right cell.

def _has_new_business_with_cell(hero, world, cell):
    """Check if the hero has something new to do on this cell."""
    if cell.items:
        return True
    if cell.cell_type == CellType.DRAGON:
        if cell.dragon_name == "Frost Wyrm" and hero.has_item("Sunblade") and world.frost_wyrm_alive:
            return True
        if cell.dragon_name == "Shadow Beast" and hero.has_item("Moonstone Lantern") and world.shadow_beast_alive:
            return True
        return False
    if cell.cell_type == CellType.INN:
        row, col = hero.position
        if (row, col) == (2, 5) and (hero.has_item("Herbal Tea") or hero.has_item("Medicine")):
            return True
        if (row, col) == (7, 4) and not world.errand_tea_picked_up:
            return True
        return False
    if cell.cell_type == CellType.SHOP:
        row, col = hero.position
        if (row, col) == (6, 2) and hero.has_item("Ember Ore") and hero.gold >= 1:
            return True
        if (row, col) == (4, 1) and hero.has_item("Moonstone") and hero.gold >= 1:
            return True
        if (row, col) == (4, 1) and hero.has_item("Dragon Scales"):
            return True
        return False
    if cell.cell_type == CellType.NPC and cell.npc:
        if cell.npc_id == "traveling_merchant" and not world.errand_letter_picked_up and not hero.has_item("Letter"):
            return True
        if cell.npc_id == "forest_witch" and hero.has_item("Ghostcaps"):
            return True
        if cell.npc_id == "old_hermit" and hero.has_item("Letter"):
            return True
        npc_name = cell.npc.name
        talked = any(npc_name in e for e in hero.journal)
        if not talked:
            return True
        return False
    return False


def _auto_interact(hero, world, cell):
    """Return an action string for obvious interactions, or None to defer.
    
    This handles the WHAT — picking up items, talking to NPCs with the right
    keywords, buying/crafting at shops, and fighting dragons.
    """
    # Always pick up items on the current cell
    if cell.items:
        return 'TOOL: pick_up()'

    if not _has_new_business_with_cell(hero, world, cell):
        return None

    if cell.cell_type == CellType.INN:
        row, col = hero.position
        if (row, col) == (2, 5) and hero.has_item("Herbal Tea"):
            return 'TOOL: talk(question="I brought herbal tea from your brother!")'
        if (row, col) == (2, 5) and hero.has_item("Medicine"):
            return 'TOOL: talk(question="I have medicine for your wife!")'
        if (row, col) == (7, 4) and not world.errand_tea_picked_up:
            return 'TOOL: talk(question="Hello! Do you need any help?")'
        return None

    if cell.cell_type == CellType.NPC and cell.npc:
        # Merchant ALWAYS gets the keyword question (even on first visit)
        if cell.npc_id == "traveling_merchant":
            return 'TOOL: talk(question="Do you have a job or errand for me?")'
        if cell.npc_id == "forest_witch" and hero.has_item("Ghostcaps"):
            return 'TOOL: talk(question="I have ghostcaps. Can you brew medicine?")'
        if cell.npc_id == "old_hermit" and hero.has_item("Letter"):
            return 'TOOL: talk(question="I have a letter to deliver to you.")'
        # First-time NPC encounter — ask for general info
        return 'TOOL: talk(question="What can you tell me about this land?")'

    if cell.cell_type == CellType.SHOP:
        row, col = hero.position
        if (row, col) == (6, 2) and hero.has_item("Ember Ore") and hero.gold >= 1:
            return 'TOOL: buy(item="Sunblade")'
        if (row, col) == (4, 1) and hero.has_item("Moonstone") and hero.gold >= 1:
            return 'TOOL: buy(item="Moonstone Lantern")'
        if (row, col) == (4, 1) and hero.has_item("Dragon Scales"):
            return 'TOOL: buy(item="sell Dragon Scales")'
        return None

    if cell.cell_type == CellType.DRAGON:
        return 'TOOL: fight()'

    return None


# ── Agentic AI Pattern 2: Memory Management ─────────────────────────────────
# Instead of dumping raw history, we build a structured memory that summarizes
# what the agent has accomplished, what failed, and what it knows about the world.

def _build_memory_summary(hero: Hero, world: GameWorld, history: list[dict], turns_used: int) -> str:
    """Build a structured memory summary from game history."""
    sections = []

    # -- Active quests FIRST --
    quests = []
    if hero.has_item("Letter"):
        quests.append("URGENT: Deliver Letter to Old Hermit at (7,1) — mention 'letter' or 'deliver' — reward: 2 gold")
    if hero.has_item("Herbal Tea"):
        quests.append("URGENT: Deliver Herbal Tea to Northern Inn at (2,5) — just talk() — reward: 2 gold")
    if hero.has_item("Ghostcaps"):
        quests.append("URGENT: Bring Ghostcaps to Forest Witch at (3,4) — mention 'medicine' or 'brew' — reward: Medicine")
    if hero.has_item("Medicine"):
        quests.append("URGENT: Bring Medicine to Northern Inn at (2,5) — just talk() — reward: Moonstone")
    if hero.has_item("Ember Ore") and not hero.has_item("Sunblade"):
        quests.append("URGENT: Craft Sunblade at Dwarf Blacksmith (6,2) — buy(item=\"Sunblade\") — costs 1 gold")
    if hero.has_item("Moonstone") and not hero.has_item("Moonstone Lantern"):
        quests.append("URGENT: Craft Moonstone Lantern at Enchanter (4,1) — buy(item=\"Moonstone Lantern\") — costs 1 gold")
    if hero.has_item("Sunblade") and world.frost_wyrm_alive:
        quests.append("URGENT: Fight Frost Wyrm at (2,1) with Sunblade — fight() — reward: 3 gold + Dragon Scales")
    if hero.has_item("Moonstone Lantern") and world.shadow_beast_alive:
        quests.append("URGENT: Fight Shadow Beast at (4,6) with Moonstone Lantern — fight() — reward: 3 gold + Dragon Scales")
    if hero.has_item("Dragon Scales"):
        quests.append("Sell Dragon Scales at Enchanter (4,1) — buy(item=\"sell Dragon Scales\") — reward: 1 gold each")
    if quests:
        sections.append("=== ACTIVE QUESTS (do these NOW) ===\n" + "\n".join(f"  >>> {q}" for q in quests))

    # -- Urgency warning --
    turns_left = 100 - turns_used
    gold_needed = 10 - hero.gold
    if turns_left <= 50 and gold_needed > 0:
        sections.append(f"*** WARNING: {turns_left} turns left, need {gold_needed} more gold! Focus on gold-earning actions! ***")

    # -- Key accomplishments --
    accomplished = []
    if world.errand_letter_picked_up:
        accomplished.append("Received Letter from Traveling Merchant")
    if world.errand_letter_delivered:
        accomplished.append("Delivered Letter to Old Hermit (+2 gold)")
    if world.errand_tea_picked_up:
        accomplished.append("Received Herbal Tea from Southern Innkeeper")
    if world.errand_tea_delivered:
        accomplished.append("Delivered Herbal Tea to Northern Inn (+2 gold)")
    if world.medicine_received:
        accomplished.append("Received Medicine from Forest Witch")
    if world.moonstone_received:
        accomplished.append("Received Moonstone from Northern Innkeeper")
    if not world.frost_wyrm_alive:
        accomplished.append("Defeated Frost Wyrm (+3 gold, +Dragon Scales)")
    if not world.shadow_beast_alive:
        accomplished.append("Defeated Shadow Beast (+3 gold, +Dragon Scales)")
    if hero.has_item("Sunblade"):
        accomplished.append("Crafted Sunblade (fire weapon — effective against Frost Wyrm)")
    if hero.has_item("Moonstone Lantern"):
        accomplished.append("Crafted Moonstone Lantern (light weapon — effective against Shadow Beast)")
    if accomplished:
        sections.append("COMPLETED:\n" + "\n".join(f"  + {a}" for a in accomplished))

    # -- Failed actions --
    failed = []
    for i, h in enumerate(history):
        if h["role"] == "result" and not h.get("success", True):
            if i > 0 and history[i-1]["role"] == "action":
                result = h["content"]
                if "nothing to pick up" in result.lower():
                    failed.append("pick_up() failed — cell was empty")
                elif "no one to talk" in result.lower():
                    failed.append("talk() failed — no NPC on that cell")
                elif "engulfs you" in result.lower() or "retreat" in result.lower():
                    failed.append("fight() failed — WRONG weapon! Frost Wyrm needs Sunblade, Shadow Beast needs Moonstone Lantern")
    if failed:
        unique_fails = list(dict.fromkeys(failed[-5:]))
        sections.append("LEARN FROM FAILURES:\n" + "\n".join(f"  ! {f}" for f in unique_fails))

    # -- Undiscovered opportunities --
    opportunities = []
    if not world.errand_letter_picked_up and not hero.has_item("Letter"):
        opportunities.append("Visit Traveling Merchant at (5,3) — ask about 'job' or 'errand' to get Letter (delivers for 2 gold)")
    if not world.errand_tea_picked_up and not hero.has_item("Herbal Tea"):
        opportunities.append("Visit Southern Inn at (7,4) — talk to innkeeper to get Herbal Tea (delivers for 2 gold)")
    if not hero.has_item("Ember Ore") and not hero.has_item("Sunblade") and world.frost_wyrm_alive:
        opportunities.append("Pick up Ember Ore at forest (2,0) — needed for Sunblade")
    if not hero.has_item("Ghostcaps") and not world.medicine_received:
        opportunities.append("Pick up Ghostcaps at forest (0,2) — needed for Medicine → Moonstone → Moonstone Lantern")
    for pos in [(5,0), (6,6), (0,7)]:
        if pos not in hero.visited:
            opportunities.append(f"Collect treasure at {pos} for 1 gold")
    if opportunities:
        sections.append("UNDISCOVERED:\n" + "\n".join(f"  ? {o}" for o in opportunities))

    return "\n".join(sections) if sections else "(no memory yet — start exploring!)"


# ── Agentic AI Pattern 3: Structured Prompting with Few-Shot Examples ────────

SYSTEM_PROMPT_TEMPLATE = """You are an AI agent controlling a hero in an 8x8 RPG grid world.

GOAL: Collect 10 gold and survive (hearts > 0). You have at most 100 turns total.

AVAILABLE TOOLS:
{tools}

CRITICAL RULES:
- NEVER call look() or status() — they run automatically and WASTE a turn.
- NEVER call move() — navigation is handled automatically for you.
- Output exactly ONE action per turn in the format shown below.
- Fight dragons ONLY with the correct weapon: Sunblade for Frost Wyrm, Moonstone Lantern for Shadow Beast.
- If you have ACTIVE QUESTS, prioritize completing them.
- Do NOT talk to an NPC you've already talked to unless you have a NEW item to deliver.

Your job is to DECIDE what to do, not to navigate. Choose from: talk, pick_up, buy, use, fight, rest.
If none of these apply, output EXPLORE to let the navigation system move you toward something useful.

OUTPUT FORMAT — first write your reasoning, then one TOOL line:

REASONING: I'm at the Traveling Merchant. I should ask about a delivery job.
TOOL: talk(question="Do you have a delivery job for me?")

REASONING: I see items on this cell. I should pick them up.
TOOL: pick_up()

REASONING: I have the Sunblade and I'm at the Frost Wyrm. Time to fight!
TOOL: fight()

REASONING: Nothing to do on this cell, I should explore.
TOOL: EXPLORE

NPC INTERACTION TIPS:
- To get the Letter from the Traveling Merchant, ask about a "job", "errand", "task", or "delivery".
- To deliver the Letter to the Old Hermit, mention "letter" or "deliver".
- To get Medicine from the Forest Witch, mention "ghostcap", "medicine", "brew", or "mushroom".
- Innkeepers give/receive items automatically when you talk() to them — just say hello.

STRATEGY:
- Pick up items immediately when you see them on a cell.
- Talk to every NPC/innkeeper you encounter — they trigger quests and rewards.
- Follow your ACTIVE QUESTS — delivering items and fighting dragons gives the most gold.
- When nothing else to do, output EXPLORE to discover new areas."""


# ── Agentic AI Pattern 4: The Think Function ────────────────────────────────
# Architecture: autopilot handles interactions, BFS handles navigation,
# the LLM only makes high-level decisions when neither applies.

def think_llm(hero: Hero, world: GameWorld, history: list[dict]) -> str:
    """LLM-powered think function demonstrating agentic AI best practices:
    
    1. Autopilot — forces obvious interactions (pick up, talk, buy, fight)
    2. BFS navigation — deterministic pathfinding toward quest targets
    3. LLM decisions — only consulted for exploration and ambiguous situations
    4. Memory management — structured summary instead of raw history
    5. Output validation — bad LLM outputs are caught and corrected
    
    Key insight: LLMs are bad at spatial reasoning (grid navigation) but good
    at language tasks (deciding what to say, prioritizing goals). So we split
    the work: BFS handles WHERE to go, LLM handles WHAT to do when we arrive.
    """
    row, col = hero.position
    cell = world.get_cell(row, col)
    cell_desc = cell.description if cell.description else cell.cell_type.label
    turns_used = len([h for h in history if h["role"] == "action"])

    # ── Layer 1: Autopilot — handle obvious interactions ──
    # If the current cell has items, an NPC with business, a shop to buy from,
    # or a dragon to fight, do it immediately. No LLM needed.
    auto = _auto_interact(hero, world, cell)
    if auto:
        return auto

    # ── Layer 2: BFS Navigation — handle movement deterministically ──
    # If we have active quest targets (items to deliver, dragons to fight),
    # navigate toward them using BFS. The LLM is terrible at translating
    # "go to (2,5)" into the right sequence of cardinal directions.
    quest_targets = _get_quest_targets(hero, world)
    if quest_targets:
        for target in quest_targets:
            d = _bfs_next_step_llm((row, col), target, world)
            if d:
                return f'TOOL: move(direction="{d}")'

    # ── Layer 3: LLM Decision — handle ambiguous exploration ──
    # Only called when there's no obvious action AND no quest target to
    # navigate toward. This is the "what should I do next?" question.
    memory = _build_memory_summary(hero, world, history, turns_used)
    system = SYSTEM_PROMPT_TEMPLATE.format(tools=TOOLS_DESCRIPTION)

    # Safe action history builder
    history_slice = history[-16:]
    history_offset = max(0, len(history) - 16)
    action_history = []
    for i, h in enumerate(history_slice):
        if h["role"] == "action":
            global_i = history_offset + i
            if global_i + 1 < len(history) and history[global_i + 1]["role"] == "result":
                result_text = f" → {history[global_i + 1]['content'][:120]}"
            else:
                result_text = ""
            action_history.append(f"  {h['content'][:80]}{result_text}")
    action_history_text = "\n".join(action_history[-8:]) if action_history else "  (first turn)"

    journal_text = "\n".join(
        f"  - {entry[:150]}" for entry in hero.journal[-8:]
    ) if hero.journal else "  (no conversations yet)"

    visited_str = ", ".join(f"({r},{c})" for r, c in sorted(hero.visited))

    user_msg = f"""== STATUS ==
Position: ({row},{col}) | Hearts: {hero.hearts}/3 | Gold: {hero.gold}/10 | Turns used: {turns_used}/100 ({100-turns_used} remaining)
Inventory: {hero.inventory if hero.inventory else '(empty)'}

== CURRENT CELL ==
{cell_desc} (type: {cell.cell_type.value})

== MEMORY ==
{memory}

== VISITED CELLS ==
{visited_str}

== RECENT ACTIONS ==
{action_history_text}

== JOURNAL (NPC conversations) ==
{journal_text}

What should you do? If you have business on this cell (talk, pick_up, buy, fight), do it.
Otherwise output TOOL: EXPLORE to let the navigation system guide you."""

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=user_msg,
            config=genai.types.GenerateContentConfig(
                system_instruction=system,
                max_output_tokens=300,
                temperature=0.3,
            ),
        )
        text = response.text
    except Exception:
        # API error — fall back to BFS navigation
        return _bfs_navigate(hero, world)

    # ── Output validation ──
    tool_name, args = parse_tool_call(text)

    # If LLM says EXPLORE or look/status, use BFS navigation
    if tool_name in ("EXPLORE", "explore", "look", "status"):
        return _bfs_navigate(hero, world)

    # If LLM outputs a move, let BFS handle it instead (LLM is bad at directions)
    if tool_name == "move":
        return _bfs_navigate(hero, world)

    # Guard: don't re-talk to NPCs with no new business
    if tool_name == "talk" and not _has_new_business_with_cell(hero, world, cell):
        return _bfs_navigate(hero, world)

    # Guard: don't fight without the right weapon
    if tool_name == "fight" and cell.cell_type == CellType.DRAGON:
        if cell.dragon_name == "Frost Wyrm" and not hero.has_item("Sunblade"):
            return _bfs_navigate(hero, world)
        if cell.dragon_name == "Shadow Beast" and not hero.has_item("Moonstone Lantern"):
            return _bfs_navigate(hero, world)

    return text

In [36]:
oracle_fn = lambda npc, q, h: llm_oracle(npc, q, h, client)

result = play_with_llm(
    think_fn=lambda hero, world, history: think_llm(hero, world, history),
    oracle_fn=oracle_fn,
    max_turns=100,
    delay=0.5,
)
print(f"\nResult: {'Victory!' if result['won'] else 'Defeat...'} | Gold: {result['gold']} | Hearts: {result['hearts']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")


Result: Victory! | Gold: 10 | Hearts: 2 | Turns: 70
Game log saved to: game_log_victory_20260312_105223.json


In [37]:
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")

Log file: game_log_victory_20260312_105223.json
(Open the file to inspect your agent's decisions turn by turn)
